# ML-only comment features — **standalone notebook**

**No Git clone.** No subprocess to repo scripts. This file embeds the project YAML and CUDA-capable inference helpers copied from `src/comment_feature_models.py`.

**You provide:** cleaned monthly Parquet on Drive (`cleaned_monthly_chunks/<subreddit>/<YYYY-MM>.parquet`).

**Notebook writes:** VM staging under `WORKDIR` then checkpoints `comment_features_ml/` back to Drive.

**On your laptop:** copy `comment_features_ml` into `data/interim/political_forums/` and run repo script:

`scripts/merge_ml_shards_into_comment_features.py --config config/political_forums_setup.yaml`

to merge Colab ML shards with locally computed lexical fields into final `comment_features/` for downstream metrics.

Runtime: enable **GPU** when `DEVICE = "cuda"`.


## 1) Control panel — edit embedded YAML and Drive paths


In [ ]:
# -----------------------------------------------------------------------------
# MASTER CONFIG — edit before running downstream cells (no Git / no repo clone)
# -----------------------------------------------------------------------------
from __future__ import annotations
from pathlib import Path

WORKDIR = Path("/content/comment_ml_standalone")

# Google Drive: mirror cleaned_monthly_chunks layout under this folder.
DRIVE_CLEANED_MONTHLY_ROOT = "/content/drive/MyDrive/SocialAIAdoption_interim/cleaned_monthly_chunks"
DRIVE_COMMENT_FEATURES_ML_ROOT = (
    "/content/drive/MyDrive/SocialAIAdoption_interim/comment_features_ml_colab"
)

RUN_MODE = "bounded"  # "bounded" | "full"
OVERWRITE = False
SUBREDDITS = ""
MONTHS = ""
DEVICE = "cuda"
BATCH_SIZE = 128
BOUNDED_MAX_TOTAL_MONTH_FILES = 2
BOUNDED_MAX_DAYS_PER_MONTH = 10
CHECKPOINT_AFTER_RUN = True
RUN_TAG = ""

# Embedded project YAML (defaults match repo — edit model IDs inline if needed)
CONFIG_YAML = r'''
project:
  name: "SocialAIAdoption"
  description: >-
    Dump-filtered political discussion plus tech comparison (3 coding, 3 career) and
    general-question subs (answers, TooAfraidToAsk, OutOfTheLoop) for format/topic diversity,
    around ChatGPT launch.

event_window:
  start_utc: "2022-11-01T00:00:00Z"
  end_utc_exclusive: "2023-05-01T00:00:00Z"
  launch_day_utc: "2022-11-30T00:00:00Z"

subreddits:
  primary:
    - politics
    - PoliticalDiscussion
    - NeutralPolitics
    - moderatepolitics
    - Ask_Politics
    - learnprogramming
    - AskProgramming
    - CodingHelp
    - cscareerquestions
    - ITCareerQuestions
    - csMajors
    - answers
    - TooAfraidToAsk
    - OutOfTheLoop
  reserve:
    - geopolitics
    - PoliticalDebate

dataset:
  fields:
    - id
    - author
    - subreddit
    - created_utc
    - body
    - score
    - parent_id
    - link_id
    - permalink
    - edited
    - controversiality
    - distinguished
    - stickied

toxicity:
  enabled: false
  method: "vader_compound_proxy"
  note: "Set enabled true and provide Perspective API key if needed."

comment_features:
  detector_primary_model: "fakespot-ai/roberta-base-ai-text-detection-v1"
  detector_secondary_model: ""
  hostility_model: ""
  emotion_model: ""
  perplexity_model: ""

paths:
  raw_dir: "data/raw/political_forums"
  interim_dir: "data/interim/political_forums"
  processed_dir: "data/processed/political_forums"
  logs_dir: "results/logs"
  tables_dir: "results/tables"
  figures_dir: "results/figures"

'''


## 2) Dependencies


In [ ]:
!pip install -q pandas pyarrow transformers torch sentencepiece pyyaml


## 3) Preflight accelerator


In [ ]:
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())


## 4) Inference helpers + job runner (inlined project code — run once per session after Control)


In [ ]:
from __future__ import annotations

from typing import Any, Dict

import math

import pandas as pd


def resolve_inference_device(device_arg: str) -> str:
    """
    Function summary: map CLI device flag to concrete backend string in {cpu,mps,cuda}.

    Parameters:
    - device_arg: one of auto, cpu, mps, cuda.

    Returns:
    - Canonical device string usable by loaders below; cuda falls back to cpu if unavailable.
    """
    if device_arg == "cpu":
        return "cpu"
    if device_arg == "mps":
        return "mps"
    if device_arg == "cuda":
        try:
            import torch

            return "cuda" if torch.cuda.is_available() else "cpu"
        except Exception:
            return "cpu"
    if device_arg == "auto":
        try:
            import torch

            if torch.cuda.is_available():
                return "cuda"
            if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
                return "mps"
        except Exception:
            pass
        return "cpu"
    return "cpu"


def pipeline_device_argument(device: str) -> Any:
    """
    Function summary: translate canonical device string to transformers `pipeline(device=...)`.

    Parameters:
    - device: cpu, mps, or cuda.

    Returns:
    - Value accepted by `transformers.pipeline` device argument (-1 for CPU, 0 for first CUDA GPU, or "mps").
    """
    if device == "cpu":
        return -1
    if device == "mps":
        return "mps"
    if device == "cuda":
        return 0
    return -1


def maybe_load_transformers_pipeline(task: str, model_id: str, device: str) -> Any:
    """
    Function summary: lazily build a text-classification pipeline or return None on failure.

    Parameters:
    - task: Hugging Face pipeline task name.
    - model_id: model repo id or path; empty string skips load.
    - device: canonical cpu|mps|cuda.

    Returns:
    - Pipeline instance or None.
    """
    if not model_id:
        return None
    try:
        from transformers import pipeline

        dev = pipeline_device_argument(device)
        return pipeline(task, model=model_id, truncation=True, device=dev)
    except Exception:
        return None


def label_score_as_ai_probability(result: Dict[str, Any]) -> float:
    """
    Function summary: convert one text-classification label dict to normalized AI probability.

    Parameters:
    - result: dict with `label` and `score` keys from a classifier.

    Returns:
    - Probability in [0, 1] representing AI-like class probability for aligned metrics.
    """
    label = str(result.get("label", "")).lower()
    score = float(result.get("score", 0.0))
    if "human" in label or label.endswith("0"):
        return float(max(0.0, min(1.0, 1.0 - score)))
    return float(max(0.0, min(1.0, score)))


def batch_text_classification_scores(pipe: Any, texts: list[str], batch_size: int) -> list[float]:
    """
    Function summary: batched text classification returning one probability per input text.

    Parameters:
    - pipe: transformers pipeline or None.
    - texts: comment bodies (already truncated upstream if needed).
    - batch_size: batch size for pipeline.

    Returns:
    - List of floats parallel to texts; NaN on batch failure rows.
    """
    if pipe is None:
        return [float("nan")] * len(texts)
    out: list[float] = []
    for start in range(0, len(texts), max(1, int(batch_size))):
        batch = texts[start : start + max(1, int(batch_size))]
        try:
            results = pipe(batch, truncation=True, batch_size=max(1, int(batch_size)))
            if isinstance(results, dict):
                results = [results]
            out.extend([label_score_as_ai_probability(r) for r in results])
        except Exception:
            out.extend([float("nan")] * len(batch))
    return out


def batch_emotion_scores(pipe: Any, texts: list[str], batch_size: int) -> Dict[str, list[float]]:
    """
    Function summary: batched emotion multi-label classification for four target labels.

    Parameters:
    - pipe: emotion pipeline or None.
    - texts: input strings.
    - batch_size: pipeline batch size.

    Returns:
    - Dict mapping anger, fear, sadness, surprise to score lists aligned with texts.
    """
    keys = ["anger", "fear", "sadness", "surprise"]
    scores: Dict[str, list[float]] = {k: [] for k in keys}
    if pipe is None:
        for k in keys:
            scores[k] = [float("nan")] * len(texts)
        return scores
    for start in range(0, len(texts), max(1, int(batch_size))):
        batch = texts[start : start + max(1, int(batch_size))]
        try:
            results = pipe(batch, truncation=True, batch_size=max(1, int(batch_size)), top_k=None)
            if results and isinstance(results[0], dict):
                results = [[r] for r in results]
            for row_result in results:
                row_map = {k: 0.0 for k in keys}
                for item in row_result:
                    label = str(item.get("label", "")).lower()
                    value = float(item.get("score", 0.0))
                    if label in row_map:
                        row_map[label] = value
                for k in keys:
                    scores[k].append(row_map[k])
        except Exception:
            for k in keys:
                scores[k].extend([float("nan")] * len(batch))
    return scores


def init_perplexity_components(model_id: str, device: str) -> tuple[Any, Any]:
    """
    Function summary: load tokenizer and causal LM for perplexity on the requested device.

    Parameters:
    - model_id: causal LM id or empty to skip.
    - device: cpu, mps, or cuda.

    Returns:
    - (tokenizer, model) or (None, None) on failure.
    """
    if not model_id:
        return None, None
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer

        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(model_id)
        if device == "cuda":
            model = model.to("cuda")
        elif device == "mps":
            model = model.to("mps")
        else:
            model = model.to("cpu")
        model.eval()
        return tokenizer, model
    except Exception:
        return None, None


def batch_perplexity(texts: list[str], tokenizer: Any, model: Any, batch_size: int, device: str) -> list[float]:
    """
    Function summary: batched perplexity from causal LM logits with truncation and masking.

    Parameters:
    - texts: input strings.
    - tokenizer: Hugging Face tokenizer or None.
    - model: causal LM or None.
    - batch_size: micro-batch size.
    - device: cpu, mps, or cuda for tensor placement.

    Returns:
    - Per-sequence perplexity floats; NaN on errors.
    """
    if tokenizer is None or model is None:
        return [float("nan")] * len(texts)
    try:
        import torch
    except Exception:
        return [float("nan")] * len(texts)

    target = "cuda" if device == "cuda" else ("mps" if device == "mps" else "cpu")
    out: list[float] = []
    for start in range(0, len(texts), max(1, int(batch_size))):
        batch = texts[start : start + max(1, int(batch_size))]
        try:
            enc = tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128,
            )
            enc = {k: v.to(target) for k, v in enc.items()}
            with torch.no_grad():
                outputs = model(**enc, labels=enc["input_ids"])
                logits = outputs.logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = enc["input_ids"][:, 1:].contiguous()
            loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
            token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            token_losses = token_losses.view(shift_labels.size())
            if "attention_mask" in enc:
                mask = enc["attention_mask"][:, 1:].contiguous()
                seq_loss = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            else:
                seq_loss = token_losses.mean(dim=1)
            batch_ppl = torch.exp(seq_loss).detach().cpu().tolist()
            out.extend([float(v) for v in batch_ppl])
        except Exception:
            out.extend([float("nan")] * len(batch))
    return out


def append_ml_columns_to_records(
    records: list[Dict[str, Any]],
    texts: list[str],
    model_config: Dict[str, str],
    device: str,
    batch_size: int,
) -> None:
    """
    Function summary: run all configured ML models and mutate records with score columns in place.

    Parameters:
    - records: list of row dicts with join keys already set; must align index-wise with texts.
    - texts: truncated comment strings for inference.
    - model_config: detector_primary, detector_secondary, hostility, emotion, perplexity ids.
    - device: canonical cpu|mps|cuda.
    - batch_size: inference batch size.

    Returns:
    - None; records are updated with ML fields and model id metadata.
    """
    detector_primary_pipe = maybe_load_transformers_pipeline(
        "text-classification", model_config["detector_primary"], device
    )
    detector_secondary_pipe = maybe_load_transformers_pipeline(
        "text-classification", model_config["detector_secondary"], device
    )
    hostility_pipe = maybe_load_transformers_pipeline("text-classification", model_config["hostility"], device)
    emotion_pipe = maybe_load_transformers_pipeline("text-classification", model_config["emotion"], device)
    ppl_tokenizer, ppl_model = init_perplexity_components(model_config["perplexity"], device)

    primary_ai = batch_text_classification_scores(detector_primary_pipe, texts, batch_size=batch_size)
    secondary_ai = batch_text_classification_scores(detector_secondary_pipe, texts, batch_size=batch_size)
    hostility_scores = batch_text_classification_scores(hostility_pipe, texts, batch_size=batch_size)
    emotion_scores = batch_emotion_scores(emotion_pipe, texts, batch_size=batch_size)
    ppl_scores = batch_perplexity(texts, ppl_tokenizer, ppl_model, batch_size=batch_size, device=device)

    for idx, record in enumerate(records):
        record["detector_primary_ai_prob"] = float(primary_ai[idx]) if idx < len(primary_ai) else float("nan")
        record["detector_primary_human_score"] = (
            float(1.0 - primary_ai[idx]) if idx < len(primary_ai) and pd.notna(primary_ai[idx]) else float("nan")
        )
        record["detector_secondary_ai_prob"] = float(secondary_ai[idx]) if idx < len(secondary_ai) else float("nan")
        record["detector_secondary_human_score"] = (
            float(1.0 - secondary_ai[idx]) if idx < len(secondary_ai) and pd.notna(secondary_ai[idx]) else float("nan")
        )
        record["hostility_score"] = float(hostility_scores[idx]) if idx < len(hostility_scores) else float("nan")
        record["emotion_anger"] = float(emotion_scores["anger"][idx])
        record["emotion_fear"] = float(emotion_scores["fear"][idx])
        record["emotion_sadness"] = float(emotion_scores["sadness"][idx])
        record["emotion_surprise"] = float(emotion_scores["surprise"][idx])
        record["perplexity"] = float(ppl_scores[idx]) if idx < len(ppl_scores) else float("nan")
        record["log_perplexity"] = (
            float(math.log(record["perplexity"])) if pd.notna(record["perplexity"]) and record["perplexity"] > 0 else float("nan")
        )
        record["detector_primary_model_id"] = model_config["detector_primary"]
        record["detector_secondary_model_id"] = model_config["detector_secondary"]
        record["hostility_model_id"] = model_config["hostility"]
        record["emotion_model_id"] = model_config["emotion"]
        record["perplexity_model_id"] = model_config["perplexity"]
        record["device_used"] = device




import shutil
import time
from datetime import datetime, timezone
from typing import Iterable

import pandas as pd
import yaml


def load_config_yaml() -> dict[str, object]:
    # Parse CONFIG_YAML from the control-cell variable.

    out = yaml.safe_load(CONFIG_YAML)
    assert isinstance(out, dict)
    return out


def interim_chunks() -> tuple[Path, Path, Path]:
    interim = WORKDIR / "data" / "interim" / "political_forums"
    return interim, interim / "cleaned_monthly_chunks", interim / "comment_features_ml"


def iter_monthly_files(
    cleaned_monthly_chunks_dir: Path,
    subreddits: list[str],
    allowed_months: set[str],
    max_month_files_per_subreddit: int,
    max_total_month_files: int,
) -> Iterable[tuple[str, Path]]:
    total = 0
    for subreddit in sorted(subreddits):
        sub_dir = cleaned_monthly_chunks_dir / subreddit
        if not sub_dir.exists():
            continue
        per_sub_count = 0
        for file_path in sorted(sub_dir.glob("*.parquet")):
            month = file_path.stem
            if allowed_months and month not in allowed_months:
                continue
            if max_month_files_per_subreddit > 0 and per_sub_count >= max_month_files_per_subreddit:
                break
            if max_total_month_files > 0 and total >= max_total_month_files:
                return
            yield subreddit, file_path
            per_sub_count += 1
            total += 1


def process_month_file_ml_nb(
    subreddit: str,
    file_path: Path,
    output_path: Path,
    overwrite: bool,
    max_days_per_month: int,
    device_arg: str,
    batch_size: int,
    model_config: dict[str, str],
) -> None:
    if output_path.exists() and not overwrite:
        print(f"[colab_ml_nb] skip_existing subreddit={subreddit} month={file_path.stem}", flush=True)
        return

    print(f"[colab_ml_nb] start subreddit={subreddit} month={file_path.stem}", flush=True)
    frame = pd.read_parquet(file_path, columns=["id", "subreddit", "date_utc", "body"])
    frame = frame[frame["subreddit"].astype("string") == subreddit].copy()
    frame["body"] = frame["body"].astype("string").fillna("")
    frame["date_utc"] = frame["date_utc"].astype("string")
    if max_days_per_month > 0 and not frame.empty:
        keep_days = sorted(frame["date_utc"].dropna().unique())[: int(max_days_per_month)]
        frame = frame[frame["date_utc"].isin(keep_days)].copy()
    if frame.empty:
        print(f"[colab_ml_nb] empty_after_filter subreddit={subreddit} month={file_path.stem}", flush=True)
        return

    records: list[dict[str, object]] = []
    texts: list[str] = []
    for row in frame.itertuples(index=False):
        text = str(getattr(row, "body", "") or "")
        records.append(
            {
                "id": str(getattr(row, "id", "") or ""),
                "subreddit": str(getattr(row, "subreddit", "") or ""),
                "date_utc": str(getattr(row, "date_utc", "") or ""),
            }
        )
        texts.append(text[:2000])

    device = resolve_inference_device(device_arg)
    print(f"[colab_ml_nb] device requested={device_arg!r} resolved={device!r}", flush=True)
    append_ml_columns_to_records(records, texts, model_config, device, batch_size)
    ml_ts = datetime.now(timezone.utc).isoformat()
    for rec in records:
        rec["ml_features_computed_at_utc"] = ml_ts

    column_order = [
        "id",
        "subreddit",
        "date_utc",
        "detector_primary_ai_prob",
        "detector_primary_human_score",
        "detector_secondary_ai_prob",
        "detector_secondary_human_score",
        "hostility_score",
        "emotion_anger",
        "emotion_fear",
        "emotion_sadness",
        "emotion_surprise",
        "perplexity",
        "log_perplexity",
        "detector_primary_model_id",
        "detector_secondary_model_id",
        "hostility_model_id",
        "emotion_model_id",
        "perplexity_model_id",
        "device_used",
        "ml_features_computed_at_utc",
    ]
    out_df = pd.DataFrame.from_records(records)
    out_df = out_df[[c for c in column_order if c in out_df.columns]]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_parquet(output_path, index=False, compression="zstd")
    print(f"[colab_ml_nb] done subreddit={subreddit} month={file_path.stem} rows={len(out_df)}", flush=True)


def sync_cleaned_from_drive(
    src_root: Path,
    dst_root: Path,
    subreddit_allowlist: set[str],
    month_allowlist: set[str],
) -> int:
    copied = 0
    dst_root.mkdir(parents=True, exist_ok=True)
    if not src_root.is_dir():
        raise NotADirectoryError(src_root)
    for sub_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        name = sub_dir.name
        if subreddit_allowlist and name not in subreddit_allowlist:
            continue
        out_sub = dst_root / name
        out_sub.mkdir(parents=True, exist_ok=True)
        for fp in sorted(sub_dir.glob("*.parquet")):
            stem = fp.stem
            if month_allowlist and stem not in month_allowlist:
                continue
            shutil.copy2(fp, out_sub / fp.name)
            copied += 1
    print(f"[colab_ml_nb] sync_cleaned copied {copied} parquet → {dst_root}")
    return copied


def sync_tree_to_drive(src: Path, dst: Path) -> int:
    if not src.exists():
        raise FileNotFoundError(src)
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for path in sorted(src.rglob("*.parquet")):
        rel = path.relative_to(src)
        out_p = dst / rel
        out_p.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, out_p)
        n += 1
    print(f"[colab_ml_nb] checkpoint copied {n} parquet → {dst}")
    return n


def run_ml_jobs(max_total_files: int, max_days: int) -> None:
    cfg = load_config_yaml()
    _, cleaned_dir, out_ml = interim_chunks()

    configured_subreddits = list(cfg["subreddits"]["primary"])  # type: ignore[index]
    if SUBREDDITS.strip():
        allow_subs = {s.strip() for s in SUBREDDITS.split(",") if s.strip()}
        configured_subreddits = [s for s in configured_subreddits if s in allow_subs]
    allow_months = {m.strip() for m in MONTHS.split(",") if m.strip()}

    feature_cfg = cfg.get("comment_features", {})
    assert isinstance(feature_cfg, dict)
    model_config = {
        "detector_primary": str(feature_cfg.get("detector_primary_model", "desklib/ai-text-detector-v1.01")),
        "detector_secondary": str(
            feature_cfg.get("detector_secondary_model", "fakespot-ai/roberta-base-ai-text-detection-v1")
        ),
        "hostility": str(feature_cfg.get("hostility_model", "unitary/unbiased-toxic-roberta")),
        "emotion": str(feature_cfg.get("emotion_model", "j-hartmann/emotion-english-distilroberta-base")),
        "perplexity": str(feature_cfg.get("perplexity_model", "gpt2")),
    }

    jobs = list(
        iter_monthly_files(
            cleaned_dir,
            configured_subreddits,
            allow_months,
            0,
            max_total_files,
        )
    )
    if not jobs:
        raise FileNotFoundError(f"No cleaned parquet under {cleaned_dir}")

    started = time.perf_counter()
    for idx, (subreddit, fp) in enumerate(jobs, start=1):
        process_month_file_ml_nb(
            subreddit,
            fp,
            out_ml / subreddit / f"{fp.stem}.parquet",
            OVERWRITE,
            max_days,
            DEVICE,
            BATCH_SIZE,
            model_config,
        )
        print(f"[colab_ml_nb] progress {idx}/{len(jobs)} elapsed_s={time.perf_counter() - started:.1f}", flush=True)


## 5) Mount Google Drive


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 6) Create VM paths — sync `cleaned_monthly_chunks` from Drive


In [ ]:
from pathlib import Path

_drive_cleaned = Path(DRIVE_CLEANED_MONTHLY_ROOT).expanduser().resolve()
_drive_ml_out = Path(DRIVE_COMMENT_FEATURES_ML_ROOT).expanduser().resolve()
drive_output_root = _drive_ml_out / RUN_TAG.strip() if RUN_TAG.strip() else _drive_ml_out
drive_output_root.mkdir(parents=True, exist_ok=True)

WORKDIR.mkdir(parents=True, exist_ok=True)
_, cleaned_local, features_ml_local = interim_chunks()
cleaned_local.mkdir(parents=True, exist_ok=True)
features_ml_local.mkdir(parents=True, exist_ok=True)

if not _drive_cleaned.exists():
    raise FileNotFoundError(f"Drive input not found: {_drive_cleaned}")

allow_subs = {s.strip() for s in SUBREDDITS.split(",") if s.strip()}
allow_months = {m.strip() for m in MONTHS.split(",") if m.strip()}

n_copied = sync_cleaned_from_drive(_drive_cleaned, cleaned_local, allow_subs, allow_months)
if n_copied == 0:
    raise RuntimeError("No parquet copied; check Drive path and SUBREDDITS/MONTHS filters.")


## 7) Run ML extraction


In [ ]:
if RUN_MODE == "bounded":
    run_ml_jobs(int(BOUNDED_MAX_TOTAL_MONTH_FILES), int(BOUNDED_MAX_DAYS_PER_MONTH))
elif RUN_MODE == "full":
    run_ml_jobs(0, 0)
else:
    raise ValueError("RUN_MODE must be bounded or full")


## 8) Upload `comment_features_ml` parquet tree to Drive


In [ ]:
_, _c, ml_out = interim_chunks()

if CHECKPOINT_AFTER_RUN:
    sync_tree_to_drive(ml_out, drive_output_root)
else:
    print("CHECKPOINT_AFTER_RUN is False; skipping Drive upload.")


## After Colab finishes (laptop)

1. Sync `comment_features_ml` from Drive → `data/interim/political_forums/comment_features_ml/` in your repo checkout.
2. Run: `.venv/bin/python scripts/merge_ml_shards_into_comment_features.py --config config/political_forums_setup.yaml`
3. (Optional) `.venv/bin/python scripts/compute_daily_repetition_similarity.py --config config/political_forums_setup.yaml`
4. `.venv/bin/python scripts/prepare_event_time_metrics.py --config config/political_forums_setup.yaml`


## 9) Verification


In [ ]:
import pandas as pd

_, _, ml_root = interim_chunks()

parquets = sorted(ml_root.rglob("*.parquet"))
if not parquets:
    raise FileNotFoundError(f"No output under {ml_root}")

df = pd.read_parquet(parquets[0])
need = {"id", "detector_primary_ai_prob", "device_used", "ml_features_computed_at_utc"}
missing = need - set(df.columns)
if missing:
    raise AssertionError(f"missing columns: {missing}")
print("sample:", parquets[0])
print("rows:", len(df))
print("device_used:", df["device_used"].iloc[0] if len(df) else None)
print("columns:", list(df.columns))
